In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\nitin\Desktop\ML Projects\Customer Churn\data\raw\Telco_customer_churn.csv")
df.head()


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Keep an untouched copy in memory
raw_df = df.copy()

print("Dataset shape:", df.shape)


In [ ]:
print("Columns:")
for index, column in enumerate(df.columns, start=1):
    print(f"{index:2}. {column}")

print("\nDataset information:")
df.info()

In [ ]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)
      .str.strip("_")
)

df.columns.tolist()

In [ ]:
object_columns = df.select_dtypes(include="object").columns

for column in object_columns:
    df[column] = df[column].str.strip()

In [ ]:
categorical_check_columns = [
    "gender",
    "senior_citizen",
    "partner",
    "dependents",
    "contract",
    "payment_method",
    "churn_label"
]

for column in categorical_check_columns:
    if column in df.columns:
        print(f"\n{column}:")
        print(df[column].value_counts(dropna=False))

In [ ]:
missing_values = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("missing_count")
)

missing_values["missing_percentage"] = (
    missing_values["missing_count"] / len(df) * 100
).round(2)

display(missing_values[missing_values["missing_count"] > 0])

In [ ]:
df["total_charges"] = pd.to_numeric(
    df["total_charges"],
    errors="coerce"
)

print(df["total_charges"].dtype)
print("Missing Total Charges:", df["total_charges"].isna().sum())

In [ ]:
missing_total_charges = df.loc[
    df["total_charges"].isna(),
    [
        "customerid",
        "tenure_months",
        "monthly_charges",
        "total_charges",
        "churn_label"
    ]
]

display(missing_total_charges)

In [ ]:
assert (
    missing_total_charges["tenure_months"] == 0
).all(), "Some missing Total Charges values have non-zero tenure."

In [ ]:
df["total_charges"] = df["total_charges"].fillna(0)

print(
    "Remaining missing Total Charges:",
    df["total_charges"].isna().sum()
)

In [ ]:
df["churn_reason"] = df["churn_reason"].fillna("Not Applicable")

print(df["churn_reason"].value_counts(dropna=False).head(10))

In [ ]:
duplicate_customer_ids = df["customerid"].duplicated().sum()
duplicate_rows = df.duplicated().sum()

print("Duplicate customer IDs:", duplicate_customer_ids)
print("Completely duplicate rows:", duplicate_rows)

In [ ]:
assert df["customerid"].is_unique, "Customer IDs are not unique."

In [ ]:
target_distribution = (
    df["churn_value"]
      .value_counts()
      .sort_index()
      .rename_axis("churn_value")
      .to_frame("customers")
)

target_distribution["percentage"] = (
    target_distribution["customers"] / len(df) * 100
).round(2)

display(target_distribution)

In [ ]:
churn_rate = df["churn_value"].mean()

print(f"Overall churn rate: {churn_rate:.2%}")

In [ ]:
pd.crosstab(
    df["churn_label"],
    df["churn_value"],
    margins=True
)

In [ ]:
expected_churn_value = (
    df["churn_label"]
      .map({"No": 0, "Yes": 1})
)

target_mismatches = (
    expected_churn_value != df["churn_value"]
).sum()

print("Target mismatches:", target_mismatches)

In [ ]:
constant_columns = [
    column
    for column in df.columns
    if df[column].nunique(dropna=False) <= 1
]

print("Constant columns:", constant_columns)

In [ ]:
target_column = "churn_value"

identifier_columns = [
    "customerid"
]

leakage_columns = [
    "churn_label",
    "churn_score",
    "churn_reason"
]

geographical_columns = [
    "city",
    "zip_code",
    "lat_long",
    "latitude",
    "longitude"
]

business_columns = [
    "cltv"
]

In [ ]:
analytics_df = df.copy()


In [ ]:
analytics_df["tenure_group"] = pd.cut(
    analytics_df["tenure_months"],
    bins=[-1, 6, 12, 24, 48, 72],
    labels=[
        "0-6 months",
        "7-12 months",
        "13-24 months",
        "25-48 months",
        "49-72 months"
    ]
)

analytics_df["monthly_charge_group"] = pd.cut(
    analytics_df["monthly_charges"],
    bins=[-np.inf, 30, 60, 90, np.inf],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

In [ ]:
display(
    analytics_df[
        [
            "tenure_months",
            "tenure_group",
            "monthly_charges",
            "monthly_charge_group"
        ]
    ].head(10)
)

In [ ]:
columns_excluded_from_model = list(
    dict.fromkeys(
        identifier_columns
        + leakage_columns
        + constant_columns
        + geographical_columns
        + business_columns
    )
)

model_df = df.drop(
    columns=columns_excluded_from_model,
    errors="ignore"
).copy()

print("Columns excluded:")
print(columns_excluded_from_model)

print("\nModel dataset shape:", model_df.shape)
display(model_df.head())

In [ ]:
X = model_df.drop(columns=[target_column])
y = model_df[target_column].astype(int)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
numeric_features = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

print("\nNumber of numeric features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

In [ ]:
assert len(df) == 7043, "Unexpected number of customers."
assert df["customerid"].is_unique, "Duplicate customer IDs found."
assert df["total_charges"].isna().sum() == 0
assert df["churn_value"].isin([0, 1]).all()
assert len(X) == len(y)
assert target_column not in X.columns
assert not set(leakage_columns).intersection(X.columns)

print("All validation checks passed.")

In [ ]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

processed_directory = project_root / "data" / "processed"
processed_directory.mkdir(parents=True, exist_ok=True)

print("Current directory:", current_directory)
print("Project root:", project_root)
print("Processed data directory:", processed_directory)

In [ ]:
analytics_path = (
    processed_directory / "telco_churn_analytics.csv"
)

model_path = (
    processed_directory / "telco_churn_model.csv"
)

analytics_df.to_csv(
    analytics_path,
    index=False
)

model_df.to_csv(
    model_path,
    index=False
)

print("Analytics dataset saved to:")
print(analytics_path)

print("\nModel dataset saved to:")
print(model_path)

In [ ]:
summary = pd.DataFrame(
    {
        "metric": [
            "Number of customers",
            "Original columns",
            "Analytics columns",
            "Model features",
            "Churned customers",
            "Overall churn rate",
            "Duplicate customer IDs",
            "Missing total charges"
        ],
        "value": [
            len(df),
            df.shape[1],
            analytics_df.shape[1],
            X.shape[1],
            int(y.sum()),
            f"{y.mean():.2%}",
            df["customerid"].duplicated().sum(),
            df["total_charges"].isna().sum()
        ]
    }
)

display(summary)